# Modern Python Development with uv

From the first chapter, we know that Python has been around for more than three decades and evolves continously. Today, most modern projects use a ``pyproject.toml`` file. However, many long-lived projects were created long before and still rely on familiar tools such as ``pip``, ``requirements.txt``, ``setup.py``, or ``setup.cfg``. Upgrading or refreshing always introduces some level of risk.

> *It works in production. Don't touch it.*

Now imagine it's your first day at a new company. Your team lead walks over to your desk and says:

> "We've got a small internal web service that needs to be tested from time to time. Nothing fancy—just make sure the endpoints are still responding."

Sounds simple enough.

## Start the server

In [25]:
!rm -rf $HOME/tmp/bobs-server/ && mkdir -p $HOME/tmp/bobs-server/ && cp -r $HOME/repos/ValentinTwin1206/modern-python-devops-egineering/projects/projXY_bobs_webserver/bobs_server/* $HOME/tmp/bobs-server/

In [26]:
!cd $HOME/tmp/bobs-server/ && python3 -m venv myvenv && ls -la $HOME/tmp/bobs-server

total 1640
drwxr-xr-x 3 fixcfhu fixcfhu    4096 Jul 28 07:39 .
drwxr-xr-x 7 fixcfhu fixcfhu    4096 Jul 28 07:39 ..
-rw-r--r-- 1 fixcfhu fixcfhu     912 Jul 28 07:39 README.md
-rw-r--r-- 1 fixcfhu fixcfhu 1644552 Jul 28 07:39 image.png
-rw-r--r-- 1 fixcfhu fixcfhu    7893 Jul 28 07:39 main.py
drwxr-xr-x 5 fixcfhu fixcfhu    4096 Jul 28 07:39 myvenv
-rw-r--r-- 1 fixcfhu fixcfhu     211 Jul 28 07:39 requirements.txt
-rw-r--r-- 1 fixcfhu fixcfhu     609 Jul 28 07:39 setup.py


In [29]:
!cd $HOME/tmp/bobs-server/ && myvenv/bin/python3 -m pip install -r requirements.txt

  Using cached requests-2.0.0-py2.py3-none-any.whl.metadata (28 kB)
  Using cached urllib3-1.7.1-py3-none-any.whl
  Using cached certifi-2015.04.28-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached chardet-2.1.1-py3-none-any.whl
  Using cached MarkupSafe-0.23.tar.gz (13 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached Jinja2-2.7.3.tar.gz (378 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached Twisted-15.5.0.tar.bz2 (3.1 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [20 lines of output]
      Traceback (most recent call last):
        File "/home/fixcfhu/tmp/bobs-server/myvenv/lib/python3.12/site-packages

In [28]:
!cd $HOME/tmp/bobs-server/ && myvenv/bin/python3 main.py

Traceback (most recent call last):
  File "/home/fixcfhu/tmp/bobs-server/main.py", line 9, in <module>
    import imp  
    ^^^^^^^^^^
ModuleNotFoundError: No module named 'imp'


##### Conclusion

Setting up the server locally is not that easy because the server itselfs rely on ``Python3.9`` and is using libraries that are not compatible with our system wide installed Python.

What options do we have?
* seperating each project with their own oci (docker) containers
* installing Python3.9 on the system

## Setup Python 3.9

There are multiple ways how to setup Python on a running system. One option is to install it via an ppa remote. It supports the installation of multiple Python versions side-by-side on the system. However the installation at least requires some basic system knowledge. 

The official [instructions](https://launchpad.net/~deadsnakes/+archive/ubuntu/ppa) are offering the following guide

```
This PPA can be added to your system manually by copying the lines below and adding them to your system's software sources.

Display sources.list entries for: 
Noble (24.04)
deb https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble main 
deb-src https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble main 
Signing key:
4096R/F23C5A6CF475977595C89F51BA6932366A755776 (What is this?)
Fingerprint:
F23C5A6CF475977595C89F51BA6932366A755776
```

```bash
# get the signing key and convert it in apt's keyring binary format
curl -fsSL "https://keyserver.ubuntu.com/pks/lookup?op=get&search=0xF23C5A6CF475977595C89F51BA6932366A755776" | gpg --dearmor | sudo tee /etc/apt/keyrings/deadsnakes.gpg >/dev/null
```

```bash
# setup the remotes
cat <<EOF | sudo tee /etc/apt/sources.list.d/deadsnakes.list
deb [trusted=yes signed-by=/etc/apt/keyrings/deadsnakes.gpg] https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble main
deb-src [trusted=yes signed-by=/etc/apt/keyrings/deadsnakes.gpg] https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble main
EOF
```

```bash
# install python3.9
apt install python3.9 python3.9-venv python3.9-pip
```

In [ ]:
!cd $HOME/tmp/bobs-server/ && rm -f myvenv && python3.9 -m venv myvenv && myvenv/bin/python3 -m pip install -r requirements.txt

In [ ]:
!cd $HOME/tmp/bobs-server/ && myvenv/bin/python3 main.py

---

Now let's try to pin the Python interpreter to version 3.10 

---

In [ ]:
!cd $HOME/tmp/demo-project && uv python pin 3.10 --verbose

##### Conclusion

Installing Python3.9 system-wide and spawning the server through a virtual environment at least work. However, we have seen that the installation is a bit tricky and requires basic system knowledge. It also seems a bit intransparent, and it is also loosly coupled from the server.

## Setup a Docker container